# Bilderkennung mit CNNs

Jetzt wollen wir für den Gestensatz einen Classifier mit State of the Art Performance trainieren. Dafür laden wir zuerst den Datensatz wie zuvor ein. CNNs können mit weniger Parameter gut mit größeren Bildern umgehen, daher starten wir direkt mit einer Größe von 96x96.

In [8]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

BATCH_SIZE = 64
IMG_SIZE = (96, 96)
directory = "../UB6/Dataset/"
train_dataset = image_dataset_from_directory(directory=directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.2,
                                             subset='training',
                                             seed=42)
validation_dataset = image_dataset_from_directory(directory=directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.2,
                                             subset='validation',
                                             seed=42)

Found 2071 files belonging to 10 classes.
Using 1657 files for training.
Found 2071 files belonging to 10 classes.
Using 414 files for validation.


## Das CNN

Jetzt wollen wir ein Convolutional Neural Network erstellen. Das funktioniert sehr ähnlich, wie zuvor, nur dass wir zwischen "Rescaling" und "Flatten" noch ein paar Layer einfügen. Zum Beispiel 3 mal je eine 2D Convolution gefolgt von einem Max Pooling Layer:
- `tf.keras.layers.Conv2D`:
    - `filters`: Wie viele Output Channel wollen wir haben (tendenziell mindestens soviele wie reinkamen)
    - `kernel_size`: Wieviele Pixel die Convolution betrifft (meistens so zwischen 3 und 11)
    - `activation`: Die Activation Function (analog zu anderen Layern)
    - `padding`: In der Regel entweder "valid" oder "same". Mit "same" bleibt die Output Höhe und Breite gleich (oft ist das ein guter Default zum Starten).
- ` tf.keras.layers.MaxPool2D`:
    - `pool_size`: Wie viele Pixel in jeder Richtung zusammengefasst werden. Um diesen Faktor verkleinert sich das Bild.
    
Dieses neuronale Netz können wir genauso trainieren wie zuvor. Je tiefer das Netz wird, umso wackeliger wird Stochastic Gradient Descent. Eine Erweiterung davon ist der der Adam Optimizer (`tf.optimizers.Adam`). Dieser macht läuft auch den Gradienten entlang, ist aber etwas stabiler (indem er sich merkt, in welche Richtung er zuvor um wieviel gelaufen ist). Auch hier geben wir die Lernrate mit (und können eine mit der Zeit sinkende Lernrate angeben).

Mit `model.summary()` können Sie sich auch wieder ausgeben lassen, wie dein neuronales Netzwerk aussieht. Je nach Definition der Layer kommen wir hier mit deutlich weniger Parametern aus, als in der Übung zuvor. Trotzdem brauchen wir eventuell mehr Parameter, als wir Trainingsbeispiele haben.

Nach ein paar Runden experimentieren bin ich auf rund 92% Accuracy auf dem Validation Set gekommen. Damit sind wir schon einen guten Schluck besser, als die SVMs auf den gleichen Daten. Mit mehr Fine Tuning ist hier sicher auch noch mehr möglich.

In [12]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(96, 96, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(filters = 96, kernel_size= 5, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(filters = 96, kernel_size= 5, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(filters = 96, kernel_size= 5, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=10, #jedes neuron bekommt 784 features und 1 bias. Bias(b) ist intercept. allgemein für logistische regression: z=w1x1+w2x2+wnxn+b -> activation function -> output
                           activation= "softmax",
                             kernel_regularizer= tf.keras.regularizers.L2(0.01) #model kriegt penalty für sehr hohe weights, deshalb werden sie gleichmäßiger verteilt (quadratische Regularisierung)
    )
])

# ADD CODE HERE

model.compile( 
    optimizer = tf.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = [tf.metrics.SparseCategoricalAccuracy]
)
# ADD CODE HERE

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_4 (Rescaling)         │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 92, 92, 96)     │         7,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 46, 46, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 42, 42, 96)     │       230,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 21, 21, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 17, 17, 96)     │       230,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 96)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 6144)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │        61,450 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 529,738 (2.02 MB)

 Trainable params: 529,738 (2.02 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# ADD CODE HERE
epochs = 100

history = model.fit(
    train_dataset,
    validation_data = validation_dataset,
    epochs = epochs
)

Epoch 1/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 22s 830ms/step - loss: 2.4527 - sparse_categorical_accuracy: 0.1074 - val_loss: 2.3928 - val_sparse_categorical_accuracy: 0.0966
Epoch 2/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 25s 944ms/step - loss: 2.3683 - sparse_categorical_accuracy: 0.1207 - val_loss: 2.3380 - val_sparse_categorical_accuracy: 0.0966
Epoch 3/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step - loss: 1.8021 - sparse_categorical_accuracy: 0.3989 - val_loss: 1.0097 - val_sparse_categorical_accuracy: 0.6836
Epoch 4/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - loss: 0.8343 - sparse_categorical_accuracy: 0.7435 - val_loss: 0.7436 - val_sparse_categorical_accuracy: 0.7754
Epoch 5/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - loss: 0.6089 - sparse_categorical_accuracy: 0.8328 - val_loss: 0.5681 - val_sparse_categorical_accuracy: 0.8575
Epoch 6/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - loss: 0.4731 - sparse_categorical_accuracy: 0.8799 - val_loss: 0.4701 - val_sparse_categorical_accuracy: 0.8816
Epoc

KeyboardInterrupt: 

## Transfer Learning

Was machen wir, wenn wir weniger Daten zur Verfügung haben? Simulieren können wir das, indem wir die größe unseres Validation Sets auf 80% erhöhen:

In [ ]:
train_dataset = image_dataset_from_directory(directory=directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.8,
                                             subset='training',
                                             seed=42)
validation_dataset = image_dataset_from_directory(directory=directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.8,
                                             subset='validation',
                                             seed=42)

Wir wollen nun mit einem vortrainierten Neuronalen Netzwerk starten. `tf.keras.applications.EfficientNetB0` implementiert das kleinste EfficientNet und hat bereits auf "imagenet" vortrainierte Gewichte. Mit wir können uns die Struktur davon direkt anschauen:

In [ ]:
tf.keras.applications.EfficientNetB0(include_top=True,
                                     weights='imagenet').summary()

Dem Konstruktor von `EfficientNetB0` können wir mit `input_shape=` auch einen anderen Input Shape mitgeben. Meistens sind nur bestimmte Kombinationen möglich - das ist hier auch der Grund, weswegen wir unseren Input als 96x96 einlesen, was am nächsten am Original Input von 100x100 ist. Da wir als Output nicht die 1000 Klassen von ImageNet, sondern unsere eigenen 10 Klassen haben, sollten wir `include_top` ausschalten.

In [ ]:
base_model = # ADD CODE HERE

base_model.summary()

Wenn wir die Summaries vergleichen, so sehen wir, dass ohne "top" 3 Layer am Ende fehlen. Für diese können wir nun unsere eigenen hinzufügen. Leider können wir nun nicht mehr "Sequential" nehmen, um das Modell zusammenzubauen, sondern müssen die etwas flexiblere "Functional API" von Tensorflow verwenden. Damit wird es aber auch nicht deutlich schwieriger.

In der Functional API definieren wir welche Inputs wir haben und wie die Outputs aus dem Input entstehen.
Hier ein Beispiel für 2 lineare Layer mit Functional API:

In [ ]:
inputs = tf.keras.Input(shape= (1234))
x = tf.keras.layers.Dense(20, activation="relu")(inputs)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

Analog können wir nun unser Modell definieren.

- Als erstes muss der Input in `tf.keras.applications.efficientnet.preprocess_input` gegeben werden. Die Funktion sorgt dafür, dass die Bilder die Annahmen von EfficientNet erfüllen (zum Beispiel die Pixel zwischen 0 und 1 skaliert werden).
- Nun wenden wir unser `base_model` an.
- Jetzt kommen die 3 Layer, die wir ersetzen müssen:
 - `tf.keras.layers.GlobalAveragePooling2D` - Das ist eine Alternative zu "Flatten", die einfach das Maximum je Channel nimmt.
 - `tf.keras.layers.Dropout` (mit Parameter 0.2) - Dropout ist eine Regularisierungstechnik für neuronale Netze
 - `tf.keras.layers.Dense` - der Output, den wir produzieren wollen
 
Bevor wir nun das Modell trainieren müssen wir Tensorflow sagen, dass die Parameter von `base_model` nicht verändert werden sollen. Das tun wir, indem wir den Parameter `.trainable` von `base_model` auf `False` setzen.

Jetzt trainieren wir unser Modell analog wie zuvor. Da das Validation Set recht groß ist, wollen wir das vielleicht beim Training weg lassen, um das ganze zu beschleunigen. Wir können die Metriken am Ende separat mit `model.evaluate` berechnen.

Trotz weniger Daten kommen wir so auf eine Accuracy, die mit unserem SVM Modell mithalten kann. Wir können als nächstes das Modell noch ein wenig Finetunen. Dafür trainieren wir noch ein paar Epochen, wobei wir auch noch ein paar Layer vom EfficientNet anpassen.

Erstmal schauen wir, wieviele Layer das EfficientNet base_model hat:

In [ ]:
len(base_model.layers)

Nun wollen wir `.trainable` für das `base_model` auf `True` und nur für einzelne Layer auf `False` setzen. Frieren Sie so die Parameter für alle, außer die letzten 10 Layer ein.

Anschließend können Sie das Model noch ein paar (~10) Epochen weiter trainieren. Hierbei sollte die Lernrate deutlich niedriger sein. Je mehr Layer wir trainieren, umso größer ist das Risiko, dass wir nen großen Schritt in die falsche Richtung machen und umso vorsichtiger müssen wir sein. Außerdem haben wir recht wenige Daten und wollen nicht zu stark overfitten. Indem wir nur wenige Epochen trainieren sorgen wir dafür, dass wir im Zweifel nicht zu viel Schaden anrichten.

In [ ]:
# ADD CODE HERE